# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only used once at the end, in `05_evaluation.ipynb`.

In [17]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [18]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [19]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of scores (h1, h2) per example.

In [20]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[-0.1848, -0.1143],
        [-0.1771, -0.0636],
        [-0.1528, -0.0544],
        [-0.1209, -0.0621],
        [-0.1661, -0.0300]], grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss for the 2 classes (course slides "Preliminaries", slide 27). `nn.CrossEntropyLoss` applies the softmax to the raw scores itself, so the classifier has no softmax layer. Before training, the loss should be about -log(0.5) ≈ 0.69, the value of random guessing between 2 classes.

In [21]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1])
tensor(0.6781, grad_fn=<NllLossBackward0>)


## One training step 
The Adam optimizer updates the weights (course slides "Preliminaries", slide 29). One training step on the 5 examples as in the course exercise "PyTorch" (Week 1): forward pass, loss, backpropagation, update. Afterwards, the loss on the same 5 examples should be lower.

In [22]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(loss)
print(criterion(model(x), y))

tensor(0.6781, grad_fn=<NllLossBackward0>)
tensor(0.6114, grad_fn=<NllLossBackward0>)


## Batches
The data is split into batches with a `DataLoader`, as in the course exercise "Experiment Tracking": training data in batches of 32, shuffled in every epoch; validation data not shuffled, in 10 batches of 100, so that all validation batches have the same size and the average of the batch losses equals the average over all examples. `TensorDataset` pairs each row of features with its label.

In [23]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

526 10
torch.Size([32, 1200]) torch.Size([32])


## Evaluation
Loss and accuracy on a dataset, batch by batch, as in the validation of the course exercise "Experiment Tracking": the loss of each batch and the predictions (`argmax`, the class with the higher score) and labels are collected; at the end, the loss is the average of the batch losses and the accuracy is the share of correct predictions over all examples. `model.eval()` and `torch.no_grad()` as in the `test_loop` of the PyTorch tutorial "Optimizing Model Parameters". A new untrained model is tested first: it should be at about 0.69 loss and 50% accuracy.

In [24]:
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6943, accuracy: 49.10%


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the raw scores) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html.
- Random seed: fixed seed 42 as in the course exercise "Experiment Tracking" (`seed_everything(42)`); `torch.manual_seed` from the PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html.
- Optimizer and training step: Adam from the course slides "Preliminaries" (Week 1), slide 29; `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` as in the course exercise "PyTorch" (Week 1).
- Batches: `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html.
- Evaluation: collecting the batch losses, `argmax` predictions and labels, joining them with `torch.cat` and averaging the batch losses as in the validation of the course exercise "Experiment Tracking" (`validation_step` and `on_validation_epoch_end`); `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html; `np.mean` from NumPy (Harris et al., 2020).
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.